# SegOCR — Colab Experiment

A scaled training run that's actually large enough to learn character predictions. Designed to fit in a single free T4 session (~2–3 hours total).

## Why this is different from the quickstart

The quickstart trained on **500 samples for 500 iterations** — that's a smoke test, not a real run. Per the Implementation Guide §5.4 expected convergence pattern:

- Background converges around **5,000 iterations**
- Common characters (e, t, a, o…) converge around **15,000 iterations**
- Rare characters (z, q, x) converge around **40,000+ iterations**

This notebook trains on **20K samples for 25K iterations** with batch size 24 — enough to clearly see character predictions on common letters and digits.

**Setup**: `Runtime → Change runtime type → T4 GPU` (free is fine).

**Total wall time on T4**: ~15 min generation + ~2 hours training + ~5 min visualization.

## 1 · Clone + install (skip if already done)

In [ ]:
import os
if not os.path.isdir('/content/segocr'):
    !git clone https://github.com/mauryantitans/SegOCR.git /content/segocr
%cd /content/segocr
!git pull --quiet

In [ ]:
!pip install -q -e .
!pip install -q -r requirements/base.txt
!pip install -q segmentation-models-pytorch wandb

## 2 · Verify GPU

Should print `cuda` and a GPU name like `Tesla T4` or `A100`. **If it prints `cpu`, switch the runtime to GPU before continuing** — training on CPU will take many hours longer.

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    name = torch.cuda.get_device_name(0)
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {name} ({mem_gb:.1f} GB)')
    # Adjust batch size based on GPU memory
    if mem_gb > 30:
        BATCH_SIZE = 32
        print('→ Using batch_size=32 (large GPU)')
    elif mem_gb > 14:
        BATCH_SIZE = 24
        print('→ Using batch_size=24 (T4-class)')
    else:
        BATCH_SIZE = 12
        print('→ Using batch_size=12 (small GPU)')
else:
    BATCH_SIZE = 4
    print('⚠️  CPU mode — training will be very slow. Switch to GPU runtime.')

## 3 · Build the experiment config

We override key parameters for a Colab-sized experiment:

- **Image size**: 256×256 (vs 512×512 default) — 4× faster, still resolves characters cleanly.
- **Encoder**: ResNet-34 — smaller than ResNet-50, more capacity than ResNet-18.
- **Char classes**: tier 1 (62 alphanumeric + bg = 63 classes).
- **Iterations**: 25K with cosine schedule + warmup.
- **Augmentation**: full augmentation enabled.
- **EMA**: enabled with decay 0.999.

In [ ]:
import yaml
from pathlib import Path

DATA_DIR = '/content/segocr_dataset'
WEIGHTS_DIR = '/content/segocr_weights'
NUM_IMAGES = 20_000
TOTAL_ITERS = 25_000

cfg = yaml.safe_load(Path('segocr/configs/default.yaml').read_text())

# ── Generator ───────────────────────────────────────────────────────────────
cfg['generator']['fonts']['root_dir'] = '/usr/share/fonts'
cfg['generator']['fonts']['cache_path'] = '/tmp/segocr_font_cache.json'
cfg['generator']['fonts']['min_size'] = 22
cfg['generator']['fonts']['max_size'] = 80
cfg['generator']['image_size'] = [256, 256]
cfg['generator']['num_workers'] = 2
cfg['generator']['output_dir'] = DATA_DIR
cfg['generator']['background']['natural_image_dirs'] = []  # tier1+2 only
# Simpler tier distribution since we don't have COCO
cfg['generator']['background']['tier_distribution'] = {
    'tier1_solid': 0.30,
    'tier2_procedural': 0.50,
    'tier3_natural': 0.10,   # falls back to procedural
    'tier4_adversarial': 0.10,
}
# Keep bundled corpora; drop the wikitext reference
cfg['generator']['text']['corpus_paths'] = [
    {'path': 'BUNDLED:signs', 'tag': 'signs', 'weight': 0.35},
    {'path': 'BUNDLED:receipts', 'tag': 'receipts', 'weight': 0.20},
    {'path': 'BUNDLED:names', 'tag': 'names', 'weight': 0.20},
    {'path': 'BUNDLED:numbers', 'tag': 'numbers', 'weight': 0.25},
]
# Bias toward easier modes for the experiment — defer hard modes for full-scale runs
cfg['generator']['layout']['modes'] = {
    'horizontal': 0.40,
    'rotated': 0.20,
    'curved': 0.10,
    'perspective': 0.10,
    'deformed': 0.05,
    'paragraph': 0.15,
}

# ── Model ───────────────────────────────────────────────────────────────────
cfg['model']['architecture'] = 'unet'
cfg['model']['encoder'] = 'resnet34'
cfg['model']['encoder_weights'] = 'imagenet'
cfg['model']['head_features'] = 64
cfg['model']['decoder_channels'] = [256, 128, 64, 32, 32]
cfg['model']['heads'] = {'semantic': True, 'affinity': True, 'direction': True}
cfg['model']['num_classes'] = 63
cfg['model']['input_size'] = [256, 256]

# ── Training ────────────────────────────────────────────────────────────────
cfg['training']['learning_rate'] = 3e-4
cfg['training']['weight_decay'] = 1e-4
cfg['training']['total_iters'] = TOTAL_ITERS
cfg['training']['warmup_iters'] = 500
cfg['training']['eval_interval'] = 2_000
cfg['training']['save_interval'] = 5_000
cfg['training']['log_interval'] = 100
cfg['training']['batch_size'] = BATCH_SIZE
cfg['training']['num_workers'] = 2
cfg['training']['mixed_precision'] = True
cfg['training']['output_dir'] = WEIGHTS_DIR
cfg['training']['ema'] = {'enabled': True, 'decay': 0.999}
cfg['training']['checkpoint_averaging'] = {'enabled': True, 'top_n': 3}
cfg['training']['keep_best_n'] = 3
cfg['training']['wandb'] = {'project': None}  # disabled by default; enable below if you want it

Path('/content/experiment_config.yaml').write_text(yaml.safe_dump(cfg))
print(f'Config written. {NUM_IMAGES} samples × {TOTAL_ITERS} iters @ batch_size={BATCH_SIZE}')

## 4 · Generate the dataset (~10–15 min)

In [ ]:
import os
if not os.path.exists(f'{DATA_DIR}/images') or len(os.listdir(f'{DATA_DIR}/images')) < NUM_IMAGES:
    !python -m scripts.generate_dataset \
        --config /content/experiment_config.yaml \
        --num-images {NUM_IMAGES} \
        --output {DATA_DIR}
else:
    print(f'Dataset already exists at {DATA_DIR} — skipping regeneration.')
    print(f'Found {len(os.listdir(f"{DATA_DIR}/images"))} images.')

## 5 · Quick look at training samples

Verify the data looks sensible before spending GPU time on it.

In [ ]:
import json
import cv2
import matplotlib.pyplot as plt
import numpy as np
import random

random.seed(42)
indices = random.sample(range(NUM_IMAGES), 8)
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, i in zip(axes.flat, indices):
    name = f'{i:06d}'
    img = cv2.cvtColor(cv2.imread(f'{DATA_DIR}/images/{name}.png'), cv2.COLOR_BGR2RGB)
    sem = cv2.imread(f'{DATA_DIR}/semantic/{name}.png', cv2.IMREAD_GRAYSCALE)
    overlay = img.copy()
    overlay[sem > 0] = [255, 0, 0]
    blended = cv2.addWeighted(img, 0.55, overlay, 0.45, 0)
    meta = json.loads(open(f'{DATA_DIR}/metadata/{name}.json').read())
    text = ''.join(c['char'] for c in meta['characters'])[:30]
    ax.imshow(blended)
    ax.set_title(f"[{meta['layout_mode']}] {text}", fontsize=9)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6 · Train (the long step — ~1.5–2 hours on T4)

If your session disconnects mid-training, the model checkpoints are written every 5,000 iterations. You can re-run this cell to start over, or open a new runtime and rerun from cell 1 (the dataset will be regenerated unless you mount Drive).

**Tip**: while this runs, you can scroll up and read the loss output to see if things are converging. By 2,000 iterations you should see the loss drop visibly. By 10,000 iterations `binary_miou` should be > 0.7 — that's the model learning to distinguish text pixels from background. Per-class IoU for common characters should start showing up around 15K iters.

In [ ]:
!python -m scripts.train_model --config /content/experiment_config.yaml

## 7 · Load the best checkpoint

In [ ]:
import torch
from pathlib import Path
from segocr.models.unet import build_model
from segocr.utils.config import load_config

cfg = load_config('/content/experiment_config.yaml')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = build_model(cfg['model']).to(device).eval()

ckpts = sorted(Path(WEIGHTS_DIR).glob('checkpoint_*.pth'))
averaged = Path(WEIGHTS_DIR) / 'averaged_best.pth'
if averaged.exists():
    print(f'Loading averaged best: {averaged}')
    state = torch.load(averaged, map_location=device)
elif ckpts:
    print(f'Loading latest: {ckpts[-1]}')
    state = torch.load(ckpts[-1], map_location=device)
else:
    raise FileNotFoundError(f'No checkpoints in {WEIGHTS_DIR}. Did training finish?')

# Use EMA weights if available, else live model weights
if 'ema' in state:
    print('→ Using EMA weights')
    model.load_state_dict(state['ema'], strict=False)
else:
    model.load_state_dict(state['model'])
model.eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model: {n_params:.1f}M params')

## 8 · Final validation metrics

Computes mIoU + per-class IoU on the held-out val split. Three mIoU variants:
- **`miou`**: mean over all 63 classes including background.
- **`fg_miou`**: mean over 62 foreground (character) classes only — the true OCR signal.
- **`binary_miou`**: collapse classes 1..62 → 1 — sanity floor and the noise-removal byproduct.

For a well-trained model on synthetic data, expect **binary_miou > 0.85**, **fg_miou > 0.4–0.6** at 25K iters.

In [ ]:
from torch.utils.data import DataLoader
from segocr.training.dataset import SegOCRDataset, collate_fn
from segocr.training.evaluator import Evaluator

val_ds = SegOCRDataset(DATA_DIR, split='val', train_aug=False)
val_loader = DataLoader(
    val_ds, batch_size=cfg['training']['batch_size'], shuffle=False,
    num_workers=2, collate_fn=collate_fn,
)
evaluator = Evaluator(num_classes=cfg['model']['num_classes'], device=device)
metrics = evaluator.evaluate(model, val_loader)

print(f"miou         {metrics['miou']:.3f}")
print(f"fg_miou      {metrics['fg_miou']:.3f}    (foreground only — the OCR signal)")
print(f"binary_miou  {metrics['binary_miou']:.3f}    (text-vs-background)")

## 9 · Per-class IoU breakdown

Reveals which characters the model has learned vs. struggles with. The Implementation Guide §6 calls this out as load-bearing — mean IoU hides per-class failures.

In [ ]:
from segocr.utils.charset import class_id_to_char

id_to_char = class_id_to_char(tier=1)
rows = []
for cid in range(1, cfg['model']['num_classes']):  # skip background
    iou = metrics.get(f'iou_class_{cid:02d}', 0.0)
    rows.append((id_to_char.get(cid, '?'), cid, iou))

# Sort by IoU descending
rows_sorted = sorted(rows, key=lambda r: -r[2])
print('Top 15 best-recognized characters:')
for char, cid, iou in rows_sorted[:15]:
    bar = '█' * int(iou * 20)
    print(f'  {char}  (class {cid:2d})  {iou:.3f}  {bar}')
print()
print('Bottom 15 worst-recognized characters:')
for char, cid, iou in rows_sorted[-15:]:
    bar = '█' * int(iou * 20)
    print(f'  {char}  (class {cid:2d})  {iou:.3f}  {bar}')

In [ ]:
# Bar chart of per-class IoU, grouped by character type
import string

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
for ax, (label, group) in zip(axes, [
    ('Uppercase A–Z', string.ascii_uppercase),
    ('Lowercase a–z', string.ascii_lowercase),
    ('Digits 0–9', string.digits),
]):
    chars = list(group)
    ious = []
    for c in chars:
        cid = next((cid for ch, cid, _ in rows if ch == c), None)
        if cid is None:
            ious.append(0)
        else:
            ious.append(metrics.get(f'iou_class_{cid:02d}', 0.0))
    ax.bar(chars, ious, color=['#2ecc71' if i > 0.3 else '#e74c3c' for i in ious])
    ax.set_ylim(0, 1)
    ax.set_title(f'{label} — per-class IoU')
    ax.axhline(0.3, color='gray', linestyle='--', alpha=0.5, label='0.30 threshold')
    ax.set_ylabel('IoU')
plt.tight_layout()
plt.show()

## 10 · Visualize predictions on validation samples

Side-by-side: input image | ground truth mask | predicted mask. Different characters show as different colors.

In [ ]:
def colorize_mask(mask, num_classes):
    '''Convert a class-id mask to a color image. Background stays black.'''
    np.random.seed(42)
    palette = np.random.randint(50, 256, size=(num_classes, 3), dtype=np.uint8)
    palette[0] = (0, 0, 0)
    return palette[mask]

n_show = 6
fig, axes = plt.subplots(n_show, 3, figsize=(12, 4 * n_show))
with torch.no_grad():
    for row in range(n_show):
        sample = val_ds[row]
        x = sample['image'].unsqueeze(0).to(device)
        out = model(x)
        pred = out['semantic'].argmax(dim=1).cpu().numpy()[0]
        gt = sample['targets']['semantic'].cpu().numpy()

        # Recover RGB image (de-normalize ImageNet stats)
        img = sample['image'].permute(1, 2, 0).cpu().numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = (np.clip(img, 0, 1) * 255).astype(np.uint8)

        axes[row, 0].imshow(img)
        axes[row, 0].set_title('Input', fontsize=10)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(colorize_mask(gt, cfg['model']['num_classes']))
        axes[row, 1].set_title(f'Ground truth ({(gt > 0).sum()} fg pixels)', fontsize=10)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(colorize_mask(pred, cfg['model']['num_classes']))
        axes[row, 2].set_title(f'Prediction ({(pred > 0).sum()} fg pixels)', fontsize=10)
        axes[row, 2].axis('off')
plt.tight_layout()
plt.show()

## 11 · Decode predicted text (simple reading-order recovery)

Take a few horizontal-layout val samples and turn the predicted mask into a string by:
1. Connected-component analysis on the predicted mask.
2. For each component, pick the most common class ID.
3. Sort components left-to-right by centroid x.
4. Concatenate.

This is a simplified version of the full post-processing pipeline (which also handles multi-line and word-segmentation).

In [ ]:
from scipy.ndimage import label as connected_components

def decode_predicted_text(pred_mask, min_area=10):
    '''Cheap reading-order recovery: connected components, sorted L→R.'''
    fg = (pred_mask > 0).astype(np.uint8)
    labeled, num = connected_components(fg)
    components = []
    for inst_id in range(1, num + 1):
        ys, xs = np.where(labeled == inst_id)
        if len(xs) < min_area:
            continue
        # Most common class ID within this component
        ids, counts = np.unique(pred_mask[ys, xs], return_counts=True)
        ids_nonzero = ids[ids > 0]
        counts_nonzero = counts[ids > 0]
        if len(ids_nonzero) == 0:
            continue
        top_class = ids_nonzero[counts_nonzero.argmax()]
        components.append((xs.mean(), top_class))
    components.sort(key=lambda c: c[0])
    return ''.join(id_to_char.get(int(cls), '?') for _, cls in components)

# Find val samples with horizontal layout for cleaner text recovery
horizontal_indices = []
for i, p in enumerate(val_ds.image_paths[:200]):
    meta = json.loads((Path(DATA_DIR) / 'metadata' / f'{p.stem}.json').read_text())
    if meta.get('layout_mode') == 'horizontal':
        horizontal_indices.append(i)
    if len(horizontal_indices) >= 8:
        break

print(f'{"Sample":<10} {"Ground truth":<25} → {"Predicted":<25} match?')
print('-' * 75)
with torch.no_grad():
    for i in horizontal_indices[:8]:
        sample = val_ds[i]
        x = sample['image'].unsqueeze(0).to(device)
        out = model(x)
        pred = out['semantic'].argmax(dim=1).cpu().numpy()[0]

        meta = json.loads((Path(DATA_DIR) / 'metadata' / f'{val_ds.image_paths[i].stem}.json').read_text())
        gt_text = ''.join(c['char'] for c in meta['characters'])
        pred_text = decode_predicted_text(pred)
        # Coarse match metric: count chars in common
        match = sum(1 for c in pred_text if c in gt_text) / max(1, len(gt_text))
        print(f'{val_ds.image_paths[i].stem:<10} {gt_text:<25} → {pred_text:<25} {match:.0%}')

## 12 · Save the model for later (optional)

Mount Google Drive and copy your trained weights so they survive the runtime shutdown.

In [ ]:
# Uncomment to mount Drive and back up weights
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p '/content/drive/MyDrive/segocr_runs/exp1'
# !cp -r {WEIGHTS_DIR}/*.pth '/content/drive/MyDrive/segocr_runs/exp1/'
# !cp /content/experiment_config.yaml '/content/drive/MyDrive/segocr_runs/exp1/'
# print('Done.')

## What to look at

If everything went well, you should see:
- **`binary_miou` > 0.8** — the model knows where text is.
- **`fg_miou` between 0.3 and 0.6** — most common characters are recognizable, rare ones aren't.
- **Most uppercase letters and digits scoring > 0.4** in the per-class chart.
- **Lowercase letters generally lower** than uppercase (smaller features, more visually similar to each other).
- **Predicted-text decoding** matching ~30–60% of characters for clean horizontal samples — character-level accuracy at this scale is far from perfect.

## What's not in this experiment

- **Bigger backgrounds**: Tier 3 (COCO) and Tier 4 (adversarial) require `bash scripts/setup_data.sh` (~25 GB download).
- **SegFormer**: more capacity than UNet but needs `mmsegmentation` (Linux/Colab) — see `requirements/train.txt`.
- **Domain adaptation**: CycleGAN, FDA, self-training, DANN — needed to bridge the synthetic→real gap on actual photographs.
- **Real-image evaluation**: ICDAR, COCO-Text, custom hard-cases benchmarks — also need download + post-processing pipeline.

## To go further

Increase to a paper-grade run:
- `NUM_IMAGES = 100_000`, `TOTAL_ITERS = 80_000`, `image_size = [512, 512]`
- Encoder: `resnet50` or `efficientnet-b4`
- Run on A100 (Colab Pro/Pro+) for ~12–18 hours
- Mount Drive and save checkpoints there to survive disconnects

See `docs/USAGE.md §7.2 Full pre-training run` for the canonical recipe.